In [1]:
import pretty_midi
import os
import shutil
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import glob

import pretty_midi
import numpy as np



/Users/andrewyang/miniconda3/envs/muse_client/lib/python3.13/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/andrewyang/miniconda3/envs/muse_client/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
SUSPICIOUS_RANGE = (0.35, 0.65)

def filter_midi_file(midi_path: str) -> tuple[bool, float]:
    """
    Applies the rule-based filter to a MIDI file to check for poor quantization.

    The rule discards a song if the ratio of notes on odd vs. even time steps
    falls within the SUSPICIOUS_RANGE for EVERY track in the song.

    Args:
        midi_path (str): The path to the MIDI file.

    Returns:
        bool: True if the file should be KEPT, False if it should be DISCARDED.
    """
    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        print(f"  - WARNING: Could not parse {Path(midi_path).name}. Discarding. Error: {e}")
        return False, 0.5 # Discard unparseable files

    if not midi_data.instruments:
        return False, 0.5 # Discard empty MIDI files with no tracks

    # --- The Core Logic ---
    # We assume a file is "bad" (all tracks are suspicious) until we find proof
    # to the contrary (i.e., at least one "good" track).
    all_tracks_are_suspicious = True
    # print('There are instruments')
    time_steps = []
    for instrument in midi_data.instruments:
        # A track with no notes cannot be suspicious, so it breaks the "every track" condition.
        if not instrument.notes:
            all_tracks_are_suspicious = False
            break

        odd_ticks = 0
        even_ticks = 0

        for note in instrument.notes:            # Convert note's start time from seconds to an integer "time step" (tick)
            time_step = midi_data.time_to_tick(note.start) // 10
            time_steps.append(time_step)
            if time_step % 2 == 0:
                even_ticks += 1
            else:
                odd_ticks += 1

        # Avoid division by zero. A track with no even notes has an infinite ratio,
        # which is well outside the suspicious range, so it's a "good" track.
        if even_ticks == 0:
            is_suspicious = False
        else:
            # ratio = odd_ticks / (odd_ticks + even_ticks)
            ratio = odd_ticks / even_ticks
            # Check if the ratio falls within the "suspicious" range
            if SUSPICIOUS_RANGE[0] <= ratio <= SUSPICIOUS_RANGE[1]:
                is_suspicious = True
            else:
                is_suspicious = False

        # If we find even one track that is NOT suspicious, the whole file is good.
        if not is_suspicious:
            all_tracks_are_suspicious = False
            break # No need to check other tracks

    # A file is discarded ONLY IF all of its tracks were suspicious.
    # Therefore, we KEEP the file if `all_tracks_are_suspicious` is False.
    return not all_tracks_are_suspicious, ratio#, time_steps

In [22]:
path = 'try_align/messy_midi/000125_0.mid'
sus, ratio = filter_midi_file(path)
print('messy_midi:')
print(sus, ratio)

path = 'try_align/quantized_librosa/librosa_000125_0.mid'
sus, ratio = filter_midi_file(path)
print('librosa:')
print(sus, ratio)

path = 'try_align/quantized_madmom/madmom_000125_0.mid'
sus, ratio = filter_midi_file(path)
print('madmom:')
print(sus, ratio)

messy_midi:
True 0.9720930232558139
librosa:
True 1.0287081339712918
madmom:
True 0.9272727272727272


In [23]:
path = 'try_align/messy_midi/000862_0.mid'
sus, ratio = filter_midi_file(path)
print('messy_midi:')
print(sus, ratio)

path = 'try_align/quantized_librosa/librosa_000862_0.mid'
sus, ratio = filter_midi_file(path)
print('librosa:')
print(sus, ratio)

path = 'try_align/quantized_madmom/madmom_000862_0.mid'
sus, ratio = filter_midi_file(path)
print('madmom:')
print(sus, ratio)

messy_midi:
True 2.6970954356846475
librosa:
True 0.8073022312373225
madmom:
True 2.2282608695652173
